In [64]:
T = CartanType(['B', 4])
W = WeylGroup(T,prefix="s")
[s1, s2, s3, s4] = W.simple_reflections()

WCR = WeylCharacterRing(T, style="coroots", base_ring=QQ)
R = WeightRing(WCR)
# EXPI MODIFIED, THIS CODE ONLY WORKS IN TYPE A
f = [w.coerce_to_sl() for w in WCR.fundamental_weights()]

n = len(W.simple_reflections())
e = s1^2
w0 = W.long_element()
S.<q> = LaurentPolynomialRing(QQ)
KL = KazhdanLusztigPolynomial(W,q)
q = 7

def Iw(w):
    a = []
    for i in range(1, n+1):
        if w.has_right_descent(i):
            a.append(i)
    return a

def Iwc(w):
    a = []
    for i in range(1, n+1):
        if not w.has_right_descent(i):
            #a.append(W.simple_reflections()[i])
            a.append(i)
    return a

def expI(l):
    a = R.one()
    for i in l:
        a = a * R(f[i-1])
    return a

def Cpsub_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(y)
        s += int(KL.P(y, w)(1)) * a
    return s

def Cpsup_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w0*w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(w0*y)
        s += int(KL.P(y, w0*w)(1)) * a
    return s

def fsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)))

def qfsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)).scale(q))

def fsup(w):
    return Cpsup_on_wr_element(w, expI(Iw(w)))

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

def Iweight(w):
    wgt = 0
    for a in Iw(w):
        wgt += f[a-1]
    return wgt

rho = Iweight(w0)

def weightpairing(l):
    a = WCR.dot_reduce(l - rho)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def woke_pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

d_sub = {}
d_sup = {}

for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_sub[i] = fsub(W[i])
    d_sup[i] = fsup(W[i])

print("Setup done, f_w, f^w are defined.")

Setup done, f_w, f^w are defined.


In [65]:
def build_matrix():
    m = []
    for i in range(len(W)):
        print(i + 1, "out of", len(W), end='\r')
        r = []
        for j in range(len(W)):
            r.append(woke_pairing(d_sub[i], d_sup[j]))
        m.append(r)
    return m
bigmat = build_matrix()
#bigmat[i][j] = woke_pairing(d_sub[i], d_sup[j]) for all i, j
alls = list(range(len(W)))
pops = list(range(len(W)))
order = []
while len(pops) > 0:
    for a in pops:
        good = True
        for b in pops:
            if b != a and bigmat[b][a] != 0:
                good = False
        if good:
            order.append(a)
            pops.remove(a)

order.reverse()
print("Computed the matrix of pairings and the upper triangular order.")

Computed the matrix of pairings and the upper triangular order.


In [66]:
def lc(i):
    a = bigmat[i][i]
    d = dict(WCR(a))
    if len(d) != 1:
        print("LC ERROR")
    else:
        return list(d.values())[0]

counter = 0
d_dual = {}
for j in order:
    counter += 1
    print(counter, "out of", len(W), end='\r')
    d_dual[j] = d_sup[j]/lc(j)
    for i in order:
        if  i != j and woke_pairing(d_sub[i], d_dual[j]) != 0:
            d_dual[j] = d_dual[j] - woke_pairing(d_sub[i], d_dual[j])*d_sup[i]/lc(i)
print("Computed the dual basis as the dictionary d_dual.")

Computed the dual basis as the dictionary d_dual.


In [5]:
#OPTIONAL CELL: CHECKING THE DUAL BASIS IS VALID.
valid = True
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    for j in range(len(W)):
        a = woke_pairing(d_sub[i], d_dual[j])
        if a != 0:
            if i != j or a != 1:
                print("ISSUE AT", i, j)
                valid = False
if valid:
    print("Checked dual basis, all is good.")
else:
    print("ISSUES! Dual basis is NOT correct.")

Checked dual basis, all is good.


In [67]:
#Defining [q]^*f_w
q = 29

d_qsub = {}
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_qsub[i] = qfsub(W[i])

In [68]:
#DEPENDS ON THE TYPE! Set the minimal Duflo involution in each left cell. Leave lcell_dict = {} in Type A!
lcell_dict = {s1*s2*s3*s4*s3*s2*s1: s1, s2*s3*s4*s3*s2: s2, s3*s4*s3: s3, s4*s3*s4: s4, s3*s4*s3*s1: s3*s1, s4*s3*s4*s1: s4*s1, s4*s2*s3*s4*s1*s2: s4*s2, s2*s3*s4*s3*s1*s2: s2*s3*s1*s2, s3*s4*s2*s3*s4*s1*s2*s3: s3*s4*s2*s3, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4: s4*s3*s4*s2*s3*s4, s4*s3*s4*s2*s3*s4*s2*s3: s4*s3*s4*s3, s4*s3*s4*s2*s3*s4*s3*s2: s4*s2*s3*s4*s3*s2, s2*s3*s4*s2*s3*s2: s3*s4*s2*s3*s4*s2*s3*s2, s4*s3*s4*s1*s2*s3*s4*s3*s2*s1: s4*s1*s2*s3*s4*s3*s2*s1, s1*s2*s3*s4*s2*s3*s2*s1: s3*s4*s1*s2*s3*s4*s2*s3*s2*s1, s1*s2*s3*s4*s3*s1*s2*s1: s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1, s1*s2*s3*s4*s1*s2*s3*s1*s2*s1: s1*s2*s3*s1*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s3*s1*s2*s1: s4*s1*s2*s3*s4*s1*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2*s1: s3*s4*s1*s2*s3*s4*s1*s2*s3*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2: s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2, s4*s3*s4*s1*s2*s3*s4*s2*s3*s1: s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s1, s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2: s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2: s4*s3*s4*s2*s3*s4*s2*s3*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2*s1: s4*s3*s4*s1*s2*s3*s4*s2*s3*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1: s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2*s1: s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2*s1}

for a in invols:
    if a not in lcell_dict:
        lcell_dict[a] = a
print("lcell_dict is defined.")
print(lcell_dict)

#list of two-sided cells, each of which is a list of left cells inside it, each of which is a list of Weyl group elements.
cells = [[[1]], [[s1, s2*s1, s3*s2*s1, s4*s3*s2*s1, s3*s4*s3*s2*s1, s2*s3*s4*s3*s2*s1, s1*s2*s3*s4*s3*s2*s1], [s2, s1*s2, s3*s2, s4*s3*s2, s3*s4*s3*s2, s2*s3*s4*s3*s2, s1*s2*s3*s4*s3*s2], [s3, s2*s3, s4*s3, s1*s2*s3, s3*s4*s3, s2*s3*s4*s3, s1*s2*s3*s4*s3], [s4, s3*s4, s2*s3*s4, s4*s3*s4, s1*s2*s3*s4]], [[s3*s1, s2*s3*s1, s4*s3*s1, s3*s4*s3*s1, s4*s2*s3*s1, s2*s3*s4*s3*s1, s3*s4*s2*s3*s1, s4*s3*s4*s2*s3*s1], [s4*s1, s3*s4*s1, s4*s2*s1, s2*s3*s4*s1, s3*s4*s2*s1, s4*s3*s4*s1, s4*s2*s3*s4*s1, s4*s3*s4*s2*s1, s3*s4*s2*s3*s4*s1, s4*s3*s4*s2*s3*s4*s1], [s4*s2, s3*s4*s2, s4*s1*s2, s3*s4*s1*s2, s4*s3*s4*s2, s2*s3*s4*s1*s2, s4*s3*s4*s1*s2, s4*s2*s3*s4*s1*s2, s3*s4*s2*s3*s4*s1*s2, s4*s3*s4*s2*s3*s4*s1*s2], [s3*s1*s2, s2*s3*s1*s2, s4*s3*s1*s2, s3*s4*s3*s1*s2, s4*s2*s3*s1*s2, s2*s3*s4*s3*s1*s2, s3*s4*s2*s3*s1*s2, s4*s3*s4*s2*s3*s1*s2], [s4*s2*s3, s3*s4*s2*s3, s4*s1*s2*s3, s3*s4*s1*s2*s3, s4*s3*s4*s2*s3, s2*s3*s4*s1*s2*s3, s4*s3*s4*s1*s2*s3, s4*s2*s3*s4*s1*s2*s3, s3*s4*s2*s3*s4*s1*s2*s3, s4*s3*s4*s2*s3*s4*s1*s2*s3], [s4*s2*s3*s4, s3*s4*s2*s3*s4, s4*s1*s2*s3*s4, s3*s4*s1*s2*s3*s4, s4*s3*s4*s2*s3*s4, s2*s3*s4*s1*s2*s3*s4, s4*s3*s4*s1*s2*s3*s4, s4*s2*s3*s4*s1*s2*s3*s4, s3*s4*s2*s3*s4*s1*s2*s3*s4, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4]], [[s1*s2*s1, s3*s1*s2*s1, s2*s3*s1*s2*s1, s4*s3*s1*s2*s1, s3*s4*s3*s1*s2*s1, s4*s2*s3*s1*s2*s1, s2*s3*s4*s3*s1*s2*s1, s3*s4*s2*s3*s1*s2*s1], [s2*s3*s2, s1*s2*s3*s2, s4*s2*s3*s2, s1*s2*s3*s1*s2, s3*s4*s2*s3*s2, s4*s1*s2*s3*s2, s3*s4*s1*s2*s3*s2, s2*s3*s4*s1*s2*s3*s2], [s1*s2*s3*s1, s2*s3*s2*s1, s1*s2*s3*s2*s1, s4*s2*s3*s2*s1, s3*s4*s2*s3*s2*s1, s4*s1*s2*s3*s2*s1, s3*s4*s1*s2*s3*s2*s1, s2*s3*s4*s1*s2*s3*s2*s1], [s2*s3*s4*s2, s1*s2*s3*s4*s2, s4*s2*s3*s4*s2, s1*s2*s3*s4*s1*s2, s3*s4*s2*s3*s4*s2, s4*s1*s2*s3*s4*s2, s3*s4*s1*s2*s3*s4*s2, s2*s3*s4*s1*s2*s3*s4*s2], [s1*s2*s3*s4*s1, s2*s3*s4*s2*s1, s1*s2*s3*s4*s2*s1, s4*s2*s3*s4*s2*s1, s3*s4*s2*s3*s4*s2*s1, s4*s1*s2*s3*s4*s2*s1, s3*s4*s1*s2*s3*s4*s2*s1, s2*s3*s4*s1*s2*s3*s4*s2*s1], [s2*s3*s4*s2*s3, s1*s2*s3*s4*s2*s3, s4*s2*s3*s4*s2*s3, s1*s2*s3*s4*s1*s2*s3, s3*s4*s2*s3*s4*s2*s3, s4*s1*s2*s3*s4*s2*s3, s3*s4*s1*s2*s3*s4*s2*s3, s2*s3*s4*s1*s2*s3*s4*s2*s3], [s1*s2*s3*s4*s3*s1, s2*s3*s4*s2*s3*s1, s1*s2*s3*s4*s2*s3*s1, s4*s2*s3*s4*s2*s3*s1, s3*s4*s2*s3*s4*s2*s3*s1, s4*s1*s2*s3*s4*s2*s3*s1, s3*s4*s1*s2*s3*s4*s2*s3*s1, s2*s3*s4*s1*s2*s3*s4*s2*s3*s1], [s1*s2*s3*s4*s3*s1*s2, s2*s3*s4*s2*s3*s1*s2, s1*s2*s3*s4*s2*s3*s1*s2, s4*s2*s3*s4*s2*s3*s1*s2, s3*s4*s2*s3*s4*s2*s3*s1*s2, s4*s1*s2*s3*s4*s2*s3*s1*s2, s3*s4*s1*s2*s3*s4*s2*s3*s1*s2, s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2]], [[s4*s1*s2*s1, s3*s4*s1*s2*s1, s2*s3*s4*s1*s2*s1, s4*s3*s4*s1*s2*s1, s4*s2*s3*s4*s1*s2*s1, s3*s4*s2*s3*s4*s1*s2*s1], [s4*s1*s2*s3*s1, s3*s4*s1*s2*s3*s1, s2*s3*s4*s1*s2*s3*s1, s4*s3*s4*s1*s2*s3*s1, s4*s2*s3*s4*s1*s2*s3*s1, s3*s4*s2*s3*s4*s1*s2*s3*s1], [s4*s1*s2*s3*s4*s1, s3*s4*s1*s2*s3*s4*s1, s2*s3*s4*s1*s2*s3*s4*s1, s4*s3*s4*s1*s2*s3*s4*s1, s4*s2*s3*s4*s1*s2*s3*s4*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1], [s4*s1*s2*s3*s1*s2, s3*s4*s1*s2*s3*s1*s2, s2*s3*s4*s1*s2*s3*s1*s2, s4*s3*s4*s1*s2*s3*s1*s2, s4*s2*s3*s4*s1*s2*s3*s1*s2, s3*s4*s2*s3*s4*s1*s2*s3*s1*s2], [s4*s1*s2*s3*s4*s1*s2, s3*s4*s1*s2*s3*s4*s1*s2, s2*s3*s4*s1*s2*s3*s4*s1*s2, s4*s3*s4*s1*s2*s3*s4*s1*s2, s4*s2*s3*s4*s1*s2*s3*s4*s1*s2, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2], [s4*s1*s2*s3*s4*s1*s2*s3, s3*s4*s1*s2*s3*s4*s1*s2*s3, s2*s3*s4*s1*s2*s3*s4*s1*s2*s3, s4*s3*s4*s1*s2*s3*s4*s1*s2*s3, s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3]], [[s4*s3*s4*s3, s4*s2*s3*s4*s3, s3*s4*s2*s3*s4*s3, s4*s1*s2*s3*s4*s3, s3*s4*s1*s2*s3*s4*s3, s4*s3*s4*s2*s3*s4*s3, s2*s3*s4*s1*s2*s3*s4*s3, s4*s3*s4*s1*s2*s3*s4*s3, s4*s3*s4*s2*s3*s4*s2*s3], [s4*s3*s4*s3*s2, s4*s2*s3*s4*s3*s2, s3*s4*s2*s3*s4*s3*s2, s4*s1*s2*s3*s4*s3*s2, s4*s3*s4*s2*s3*s4*s2, s3*s4*s1*s2*s3*s4*s3*s2, s4*s3*s4*s2*s3*s4*s3*s2, s2*s3*s4*s1*s2*s3*s4*s3*s2, s4*s3*s4*s1*s2*s3*s4*s3*s2], [s2*s3*s4*s2*s3*s2, s4*s3*s4*s2*s3*s2, s1*s2*s3*s4*s2*s3*s2, s4*s2*s3*s4*s2*s3*s2, s1*s2*s3*s4*s1*s2*s3*s2, s3*s4*s2*s3*s4*s2*s3*s2, s4*s1*s2*s3*s4*s2*s3*s2, s3*s4*s1*s2*s3*s4*s2*s3*s2, s2*s3*s4*s1*s2*s3*s4*s2*s3*s2], [s4*s3*s4*s3*s2*s1, s4*s2*s3*s4*s3*s2*s1, s3*s4*s2*s3*s4*s3*s2*s1, s4*s1*s2*s3*s4*s3*s2*s1, s4*s3*s4*s2*s3*s4*s2*s1, s3*s4*s1*s2*s3*s4*s3*s2*s1, s4*s3*s4*s2*s3*s4*s3*s2*s1, s2*s3*s4*s1*s2*s3*s4*s3*s2*s1, s4*s3*s4*s1*s2*s3*s4*s3*s2*s1], [s2*s3*s4*s2*s3*s2*s1, s4*s3*s4*s2*s3*s2*s1, s1*s2*s3*s4*s2*s3*s2*s1, s4*s2*s3*s4*s2*s3*s2*s1, s1*s2*s3*s4*s1*s2*s3*s2*s1, s3*s4*s2*s3*s4*s2*s3*s2*s1, s4*s1*s2*s3*s4*s2*s3*s2*s1, s3*s4*s1*s2*s3*s4*s2*s3*s2*s1, s2*s3*s4*s1*s2*s3*s4*s2*s3*s2*s1], [s1*s2*s3*s4*s3*s1*s2*s1, s2*s3*s4*s2*s3*s1*s2*s1, s4*s3*s4*s2*s3*s1*s2*s1, s1*s2*s3*s4*s2*s3*s1*s2*s1, s4*s2*s3*s4*s2*s3*s1*s2*s1, s3*s4*s2*s3*s4*s2*s3*s1*s2*s1, s4*s1*s2*s3*s4*s2*s3*s1*s2*s1, s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1, s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1]], [[s4*s3*s4*s3*s1, s4*s2*s3*s4*s3*s1, s3*s4*s2*s3*s4*s3*s1, s4*s1*s2*s3*s4*s3*s1, s3*s4*s1*s2*s3*s4*s3*s1, s4*s3*s4*s2*s3*s4*s3*s1, s4*s3*s4*s1*s2*s3*s4*s3*s1, s4*s3*s4*s2*s3*s4*s2*s3*s1], [s4*s3*s4*s3*s1*s2, s4*s2*s3*s4*s3*s1*s2, s3*s4*s2*s3*s4*s3*s1*s2, s4*s1*s2*s3*s4*s3*s1*s2, s3*s4*s1*s2*s3*s4*s3*s1*s2, s4*s3*s4*s2*s3*s4*s3*s1*s2, s4*s3*s4*s1*s2*s3*s4*s3*s1*s2, s4*s3*s4*s2*s3*s4*s2*s3*s1*s2], [s4*s3*s4*s1*s2*s3*s2, s4*s2*s3*s4*s1*s2*s3*s2, s3*s4*s2*s3*s4*s1*s2*s3*s2, s4*s1*s2*s3*s4*s1*s2*s3*s2, s3*s4*s1*s2*s3*s4*s1*s2*s3*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s2, s4*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s1*s2], [s4*s3*s4*s3*s1*s2*s1, s4*s2*s3*s4*s3*s1*s2*s1, s3*s4*s2*s3*s4*s3*s1*s2*s1, s4*s1*s2*s3*s4*s3*s1*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s1, s3*s4*s1*s2*s3*s4*s3*s1*s2*s1, s4*s3*s4*s2*s3*s4*s3*s1*s2*s1, s4*s3*s4*s1*s2*s3*s4*s3*s1*s2*s1], [s4*s3*s4*s1*s2*s3*s4*s2, s4*s2*s3*s4*s1*s2*s3*s4*s2, s3*s4*s2*s3*s4*s1*s2*s3*s4*s2, s4*s2*s3*s4*s1*s2*s3*s4*s3*s2, s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2], [s4*s3*s4*s1*s2*s3*s2*s1, s4*s2*s3*s4*s1*s2*s3*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s2*s1, s4*s1*s2*s3*s4*s1*s2*s3*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s1, s3*s4*s1*s2*s3*s4*s1*s2*s3*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s2*s1, s4*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2*s1], [s4*s2*s3*s4*s1*s2*s3*s4*s3, s4*s3*s4*s1*s2*s3*s4*s2*s3, s3*s4*s2*s3*s4*s1*s2*s3*s4*s3, s4*s2*s3*s4*s1*s2*s3*s4*s2*s3, s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s3, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3], [s4*s3*s4*s1*s2*s3*s4*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s3*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s2*s1]], [[s1*s2*s3*s1*s2*s1, s4*s1*s2*s3*s1*s2*s1, s3*s4*s1*s2*s3*s1*s2*s1, s2*s3*s4*s1*s2*s3*s1*s2*s1, s4*s3*s4*s1*s2*s3*s1*s2*s1, s1*s2*s3*s4*s1*s2*s3*s1*s2*s1, s4*s2*s3*s4*s1*s2*s3*s1*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s1*s2*s1, s4*s1*s2*s3*s4*s1*s2*s3*s1*s2*s1, s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2*s1], [s1*s2*s3*s4*s1*s2*s1, s4*s1*s2*s3*s4*s1*s2*s1, s3*s4*s1*s2*s3*s4*s1*s2*s1, s2*s3*s4*s1*s2*s3*s4*s1*s2*s1, s4*s3*s4*s1*s2*s3*s4*s1*s2*s1, s2*s3*s4*s1*s2*s3*s4*s3*s1*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s3*s1*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s1*s2*s1], [s1*s2*s3*s4*s1*s2*s3*s1, s4*s1*s2*s3*s4*s1*s2*s3*s1, s3*s4*s1*s2*s3*s4*s1*s2*s3*s1, s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1, s4*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1, s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1, s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2*s1], [s1*s2*s3*s4*s1*s2*s3*s1*s2, s4*s1*s2*s3*s4*s1*s2*s3*s1*s2, s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2, s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2, s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2, s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2, s4*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2, s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2], [s2*s3*s4*s1*s2*s3*s4*s3*s1, s4*s2*s3*s4*s1*s2*s3*s4*s3*s1, s4*s3*s4*s1*s2*s3*s4*s2*s3*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s1, s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1], [s2*s3*s4*s1*s2*s3*s4*s3*s1*s2, s4*s2*s3*s4*s1*s2*s3*s4*s3*s1*s2, s4*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2, s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s1*s2, s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2, s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s1*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2]], [[s4*s3*s4*s2*s3*s4*s2*s3*s2, s4*s3*s4*s1*s2*s3*s4*s2*s3*s2, s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s2, s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2], [s4*s3*s4*s2*s3*s4*s2*s3*s2*s1, s4*s3*s4*s1*s2*s3*s4*s2*s3*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2*s1], [s4*s3*s4*s2*s3*s4*s2*s3*s1*s2*s1, s4*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s1*s2*s1, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1], [s4*s3*s4*s2*s3*s4*s1*s2*s3*s1*s2*s1, s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2*s1, s4*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2*s1]], [[s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2*s1]]]
#B3 cells = [[[s1*s1]], [[s1, s2*s1, s3*s2*s1, s2*s3*s2*s1, s1*s2*s3*s2*s1], [s2, s1*s2, s3*s2, s2*s3*s2, s1*s2*s3*s2], [s3, s2*s3, s1*s2*s3, s3*s2*s3]], [[s3*s1, s2*s3*s1, s3*s2*s3*s1], [s3*s1*s2, s2*s3*s1*s2, s3*s2*s3*s1*s2], [s3*s1*s2*s3, s2*s3*s1*s2*s3, s3*s2*s3*s1*s2*s3]], [[s1*s2*s1, s3*s1*s2*s1, s2*s3*s1*s2*s1], [s1*s2*s3*s1, s3*s1*s2*s3*s1, s2*s3*s1*s2*s3*s1], [s1*s2*s3*s1*s2, s3*s1*s2*s3*s1*s2, s2*s3*s1*s2*s3*s1*s2]], [[s3*s2*s3*s2, s3*s1*s2*s3*s2, s2*s3*s1*s2*s3*s2, s3*s2*s3*s1*s2*s3*s2, s3*s2*s3*s1*s2*s3*s1*s2], [s3*s2*s3*s2*s1, s3*s1*s2*s3*s2*s1, s2*s3*s1*s2*s3*s2*s1, s3*s2*s3*s1*s2*s3*s1, s3*s2*s3*s1*s2*s3*s2*s1], [s1*s2*s3*s1*s2*s1, s3*s2*s3*s1*s2*s1, s3*s1*s2*s3*s1*s2*s1, s2*s3*s1*s2*s3*s1*s2*s1]], [[s3*s2*s3*s1*s2*s3*s1*s2*s1]]]

lcell_dict is defined.
{s1*s2*s3*s4*s3*s2*s1: s1, s2*s3*s4*s3*s2: s2, s3*s4*s3: s3, s4*s3*s4: s4, s3*s4*s3*s1: s3*s1, s4*s3*s4*s1: s4*s1, s4*s2*s3*s4*s1*s2: s4*s2, s2*s3*s4*s3*s1*s2: s2*s3*s1*s2, s3*s4*s2*s3*s4*s1*s2*s3: s3*s4*s2*s3, s4*s3*s4*s2*s3*s4*s1*s2*s3*s4: s4*s3*s4*s2*s3*s4, s4*s3*s4*s2*s3*s4*s2*s3: s4*s3*s4*s3, s4*s3*s4*s2*s3*s4*s3*s2: s4*s2*s3*s4*s3*s2, s2*s3*s4*s2*s3*s2: s3*s4*s2*s3*s4*s2*s3*s2, s4*s3*s4*s1*s2*s3*s4*s3*s2*s1: s4*s1*s2*s3*s4*s3*s2*s1, s1*s2*s3*s4*s2*s3*s2*s1: s3*s4*s1*s2*s3*s4*s2*s3*s2*s1, s1*s2*s3*s4*s3*s1*s2*s1: s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2*s1, s1*s2*s3*s4*s1*s2*s3*s1*s2*s1: s1*s2*s3*s1*s2*s1, s4*s2*s3*s4*s1*s2*s3*s4*s3*s1*s2*s1: s4*s1*s2*s3*s4*s1*s2*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s2*s1: s3*s4*s1*s2*s3*s4*s1*s2*s3*s1, s3*s4*s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2: s2*s3*s4*s1*s2*s3*s4*s1*s2*s3*s1*s2, s4*s3*s4*s1*s2*s3*s4*s2*s3*s1: s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s3*s1, s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2: s4*s3*s4*s2*s3*s4*s1*s2*s3*s4*s2*s3*s1*s2, s4

In [12]:
def nqpair(w, y):
    #new/normalized q_pairing
    return woke_pairing(d_qsub[list(W).index(w)], d_dual[list(W).index(lcell_dict[y])])

def mw(w):
    return nqpair(w,w)

def nw(w):
    return woke_pairing(d_qsub[list(W).index(w)], d_dual[list(W).index(w)])

for c in cells:
    celltot = 0
    print(c)
    for lcell in c:
        for w in lcell:
            print(w)
            print(nw(w))
            if w * w == s1 * s1:
                print(mw(w))
            else:
                print("not invol")
        print()
    print()

[[1]]
1
B3(28,28,28)
B3(28,28,28)


[[s1, s2*s1, s3*s2*s1, s2*s3*s2*s1, s1*s2*s3*s2*s1], [s2, s1*s2, s3*s2, s2*s3*s2, s1*s2*s3*s2], [s3, s2*s3, s1*s2*s3, s3*s2*s3]]
s1
B3(0,28,28)
B3(0,28,28)
s2*s1
B3(0,28,28)
not invol
s3*s2*s1
B3(0,27,28) + B3(0,28,28)
not invol
s2*s3*s2*s1
B3(0,28,28)
not invol
s1*s2*s3*s2*s1
B3(0,28,28)
B3(0,27,28)

s2
B3(26,0,28) + B3(27,1,26) + B3(27,0,28) + B3(28,0,28)
B3(26,0,28) + B3(27,1,26) + B3(27,0,28) + B3(28,0,28)
s1*s2
B3(26,0,28) + B3(27,1,26) + B3(27,0,28) + B3(28,0,28)
not invol
s3*s2
B3(27,0,26) + B3(26,0,28) + B3(28,0,26) + 2*B3(27,1,26) + B3(27,0,28) + B3(29,0,26) + B3(28,0,28)
not invol
s2*s3*s2
B3(26,0,28) + B3(27,1,26) + B3(27,0,28) + B3(28,0,28)
B3(27,0,26) + B3(28,0,26) + B3(27,1,26) + B3(29,0,26)
s1*s2*s3*s2
B3(26,0,28) + B3(27,1,26) + B3(27,0,28) + B3(28,0,28)
not invol

s3
B3(28,27,0) + B3(28,28,0)
B3(28,27,0) + B3(28,28,0)
s2*s3
B3(28,27,0) + B3(27,28,0) + B3(29,27,0) + B3(28,28,0)
not invol
s1*s2*s3
B3(28,27,0) + B3(27,28,0) + B3(29,27,0

In [ ]:
ntotlist = []
for c in cells:
    print(c)
    mtot = 0
    ntot = 0
    for lcell in c:
        print(lcell)
        for w in lcell:
            ntot += nw(w)
            ntotlist.append(nw(w))
            if w * w == s1 * s1:
                mtot += mw(w)
    print(cure29(ntot))
    print()
    print(cure29(mtot))
    print()
    print("---")

In [ ]:
ntotlist = []
for c in cells:
    print(c)
    for lcell in c:
        mtot = 0
        ntot = 0
        print(lcell)
        for w in lcell:
            ntot += nw(w)
            ntotlist.append(nw(w))
            if w * w == s1 * s1:
                mtot += mw(w)
        print(cure29(ntot))
        print(cure29(mtot))
        print()
    print()
    print("---")

In [ ]:
R.<q> = LaurentPolynomialRing(ZZ)
H = IwahoriHeckeAlgebra('B3', q^2)
T=H.T(); Cp=H.Cp(); C=H.C()
print(C(T[3,2,3,1,2,3,1,2,1]*C[1]))

In [ ]:
ntotlist = []
wlist = []
for c in cells:
    print(c)
    ntot = 0
    for lcell in c:
        print(lcell)
        for w in lcell:
            ntot += nw(w)
            wlist.append(w)
            ntotlist.append(nw(w))
    print(cure29(ntot))
    print()
    print(cure29(mtot))
    print()
    print("---")

In [ ]:
B3 = WCR
q = 29
print(-B3(0,0,q-5) - B3(0,1,q-5) + B3(1,0,q-3) + B3(0,0,q-1) + B3(q-4,0,0) - B3(q-3,0,0) + B3(0,q-3,0) - B3(q-2,0,0) + B3(1,q-3,0) - 2*B3(0,q-3,2) + B3(q-1,0,0) + B3(1,q-2,0) + B3(0,q-1,0) + B3(q-2,0,q-3) + B3(q-3,0,q-1) + B3(q-1,0,q-3) + 2*B3(q-2,1,q-3) + B3(q-2,0,q-1) + B3(0,q-2,q-1) + B3(q,0,q-3) + B3(q-1,0,q-1) + B3(0,q-1,q-1) + B3(q-1,q-2,0) + B3(q-2,q-1,0) + B3(q,q-2,0) + B3(q-1,q-1,0))

In [ ]:
def cure29(s):
    s = str(s)
    s1 = s.replace("29","q")
    s2 = s1.replace("28","q-1")
    s3 = s2.replace("27","q-2")
    s4 = s3.replace("26","q-3")
    s5 = s4.replace("25","q-4")
    s6 = s5.replace("24","q-5")
    s7 = s6.replace("30","q+1")
    return s7
print(cure29("B3(27,0,26) + B3(26,0,28) + B3(28,0,26) + 2*B3(27,1,26) + B3(27,0,28) + B3(0,27,28) + B3(29,0,26) + B3(28,0,28) + B3(0,28,28) + B3(28,27,0) + B3(27,28,0) + B3(29,27,0) + B3(28,28,0)"))



In [ ]:
bigntot = 0
for w in W:
    bigntot += nw(w)
print(cure29(bigntot))

In [ ]:
a = 0
for b in ntotlist:
    a += b

print(cure29(a))

In [ ]:
pi1 = Mw[s3*s2*s3*s2] + Mw[s3*s1*s2*s3*s2*s1] + Mw[s1*s2*s3*s1*s2*s1]
pi2 = Mw[s3*s2*s3*s1*s2*s3*s1*s2] + Mw[s3*s2*s3*s1*s2*s3*s2*s1] + Mw[s2*s3*s1*s2*s3*s1*s2*s1]
pi3 = Mw[s3*s2*s3*s2] + Mw[s3*s1*s2*s3*s2*s1] + Mw[s2*s3*s1*s2*s3*s1*s2*s1]
pi4 = Mw[s3*s2*s3*s1*s2*s3*s1*s2] + Mw[s3*s2*s3*s1*s2*s3*s2*s1] + Mw[s1*s2*s3*s1*s2*s1]
print("print("+ cure29(pi1) + ")")
print("print("+ cure29(pi2) + ")")
print("print("+ cure29(pi3) + ")")
print("print("+ cure29(pi4) + ")")
print(cure29(3*pi1 + 2*pi3 + pi4))

In [ ]:
lcell = [s1, s2*s1, s3*s2*s1, s2*s3*s2*s1, s1*s2*s3*s2*s1]
tot = 0
for a in lcell:
    if a*a == s1*s1:
        tot += mw(a)
print(cure29(tot))